In [ ]:
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt

from tb_macro.constants import AGE_STRATA, ISO3, START_TIME, END_TIME, SOLVER_KWARGS
from tb_macro.epi import get_base_model, add_flows_to_model, initialise_pops
from tb_macro.inputs import load_demography, load_fertility, load_who_outcomes
from tb_macro.demography import prepare_pop_data_for_entries
from tb_macro.parameters import BASE_PARAMS
from tb_macro.outputs import (
    get_share_folder_file_path,
    get_age_inc,
    get_age_prev,
    get_age_latent,
    get_age_notifs,
    get_age_deaths,
    get_total_pop,
    get_posterior_samples,
    collate_output_table,
)
from tb_macro.plotting import plot_outputs

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))
add_flows_to_model(
    epi_model, 
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    death_rates,
    tsr,
    death_in_unsucc,
    entry_times,
    entry_rates,
)
initialise_pops(epi_model, disease_state, age_strat, start_apops)

In [ ]:
idata = az.from_netcdf("nuts_100_100_idata.nc")
az.summary(idata)

In [ ]:
# Load an idata that was prepared earlier
idata = az.from_netcdf("nuts_100_100_idata.nc")

In [ ]:
# Get samples
samples = get_posterior_samples(idata, 5)

In [ ]:
# Collate outputs
scen_params = [{}, {"detect_gap_reduction": 0.5}]
sample_labels = []
indicator_funcs = {
    "incidence": get_age_inc,
    "prevalence": get_age_prev,
    "latent": get_age_latent,
    "notifications": get_age_notifs,
    "deaths": get_age_deaths,
    "total_pop": get_total_pop,
}
outputs = [{out: [] for out in indicator_funcs} for _ in scen_params]

for i in range(samples.sizes["sample"]):
    run = f"chain_{int(samples['chain'][i])}/draw_{int(samples['draw'][i])}"
    sample_labels.append(run)
    c_params = {k: float(samples[k].isel(sample=i)) for k in idata.posterior.data_vars}
    for s, s_params in enumerate(scen_params):
        results = epi_model.run(BASE_PARAMS | c_params | s_params, solver_kwargs=SOLVER_KWARGS)
        for out, func in indicator_funcs.items():
            output = func(results, age_strat, disease_state).to_pandas_df()
            output.columns.name = "age_group"
            outputs[s][out].append(output)

In [ ]:
full_out = collate_output_table(outputs, sample_labels)

In [ ]:
def sum_df_over_lower_level(df):
    return df.T.groupby(level=0).sum().T

s_plot = 0
total_pop = sum_df_over_lower_level(full_out[s_plot]["total_pop"])
incs = sum_df_over_lower_level(full_out[s_plot]["incidence"])
notifs = sum_df_over_lower_level(full_out[s_plot]["notifications"])
prevs = sum_df_over_lower_level(full_out[s_plot]["prevalence"])
tb_deaths = sum_df_over_lower_level(full_out[s_plot]["deaths"])
latent = sum_df_over_lower_level(full_out[s_plot]["latent"])

In [ ]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, who_mort, latent, None, total_pop, 1980.0, 2050.0, "count")

In [ ]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, None, latent, LATENT_TARGET, total_pop, 1980.0, 2050.0, "rate")

In [ ]:
# james_work_gdrive = "/Users/jtrauer/Library/CloudStorage/GoogleDrive-james.trauer@monash.edu/"
# out_path = get_share_folder_file_path(james_work_gdrive) 
# full_out.to_csv(out_path / "full_outputs.csv")